In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
import sklearn.metrics as skm
import pickle
import boto3
from tqdm import tqdm

try:
    import catboost
except:
    ! pip install catboost

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.0' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


### Functions

In [2]:
def train_test_split(df, flt_prop_train=0.66):
    int_nrows = df.shape[0]
    df['row'] = list(range(1, int_nrows+1))
    df['row'] = df['row'] / int_nrows
    df['data_set'] = df['row'].apply(
        lambda x: 'train' if x <= flt_prop_train else 'test',
    )
    df_train = df[df['data_set']=='train'].copy()
    df_train.drop('data_set', axis=1, inplace=True)
    df_test = df[df['data_set']=='test'].copy()
    df_test.drop('data_set', axis=1, inplace=True)
    return df_train, df_test

In [3]:
# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # init client
    boto3.resource('s3').Bucket(str_project).Object(str_bucket_path).download_file(str_local_path)

In [4]:
# upload to s3
def upload_to_s3(str_local_path, str_bucket_path, str_project):
    boto3.resource('s3').Bucket(str_project).Object(str_bucket_path).upload_file(str_local_path)

### Constants

In [5]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
str_target = 'target'
str_variant = 'noPTImodel7'

Project: 20231010-gen-xii


### Import aggregated PD test data

In [6]:
str_filename = 'df.csv'
str_uri = f's3://{str_project}/08_retro_scoring/08_segment_distributions/01_gen_xii_test/{str_filename}'
list_cols = [
    'bigaccountid__app',
    'yhat_pricing_pd',
]
df = pd.read_csv(str_uri, usecols=list_cols)
df['bigaccountid__app'] = df['bigaccountid__app'].astype(int)
dict_rename = {
    'yhat_pricing_pd': 'yhat_DQ15_2',
}
df.rename(columns=dict_rename, inplace=True)
df

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:272: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


,bigaccountid__app,yhat_DQ15_2
0,2867541,0.891058
1,2867543,0.359111
2,2867556,0.657558
3,2867569,0.613629
4,2867589,0.725824
...,...,...
19768,3396623,0.165674
19769,3397156,0.067196
19770,3397435,0.112210
19771,3397518,0.441041


### Get 15DPD2months target

In [7]:
str_filename = 'GenXIIPerformanceMonitoringTarget.csv'
str_uri = f's3://{str_project}/11_monitoring/input/{str_filename}'
list_cols = [
    'bigAccountId',
    'bitDebtor',
    'DQ15_2',
]
df_tmp = pd.read_csv(
    str_uri, 
    usecols=list_cols,
)
# sort
df_tmp.sort_values(by=['bigAccountId','bitDebtor'], ascending=[True,False], inplace=True)
# agg
df_tmp = df_tmp.groupby(by='bigAccountId', as_index=False).agg({
    'DQ15_2': 'first',
})
# show
df_tmp

,bigAccountId,DQ15_2
0,1337511,0
1,1337528,0
2,1337539,0
3,1337542,0
4,1337547,0
...,...,...
154550,4812483,0
154551,4812497,0
154552,4812498,0
154553,4812503,0


### Join

In [8]:
df = pd.merge(
    left=df,
    right=df_tmp,
    left_on='bigaccountid__app',
    right_on='bigAccountId',
    how='inner',
)
df.drop('bigAccountId', axis=1, inplace=True)
# show
df

,bigaccountid__app,yhat_DQ15_2,DQ15_2
0,2867541,0.891058,1
1,2867543,0.359111,0
2,2867556,0.657558,0
3,2867569,0.613629,0
4,2867589,0.725824,0
...,...,...,...
19768,3396623,0.165674,0
19769,3397156,0.067196,0
19770,3397435,0.112210,0
19771,3397518,0.441041,0


### Get the pricing PD predictions and target

In [9]:
str_filename = 'df_test_raw.gzip'
str_uri = f's3://{str_project}/02_pricing_pd/01_data_prep/03_train_valid_test_split/{str_filename}'
df_tmp = pd.read_parquet(
    str_uri, 
)
# show
df_tmp

,dtmstampcreation__base,dtmapproved__base,dtmdeclined__base,observationdate__base,analyticsmatchkey__base,decision_dte__base,booked__base,acct_typ_cde__base,open_dte__base,curr_bal_amt__base,...,fltdowncash__app,fltapproveddowntotal__app,payment__app,dti__app,pti__app,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app
99452,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,1500.0,446.85,0.420689,0.109958,0,0.985887,auto,1,2016-02-17 12:47:19.233
99453,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1000.0,1000.0,401.91,0.389347,0.098876,1,1.126145,auto,1,2011-04-12 13:36:22.680
99454,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,500.0,500.0,316.80,0.404689,0.126737,0,1.275782,auto,0,2008-01-15 08:53:13.480
99455,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,500.0,500.0,432.25,0.418063,0.125296,0,1.101601,auto,1,2009-06-11 17:08:59.937
99456,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,303.96,0.491165,0.159419,0,1.338733,auto,0,2012-01-10 16:05:32.420
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124312,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,3500.0,573.97,0.360741,0.079448,0,0.911393,suv,1,2010-07-30 13:50:37.207
124311,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,3500.0,573.97,0.360741,0.079448,0,0.911393,suv,1,2010-07-30 13:50:37.207
124313,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,447.44,0.172283,0.090964,1,1.188462,auto,1,2013-07-24 10:29:42.193
124314,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,444.98,0.496912,0.144283,1,1.145956,auto,0,2006-09-09 11:59:48.000


In [10]:
# preprocess data
for str_filename in ['cls_model_preprocessing.pkl','preprocessing.py']:
    str_bucket_path = f'01_ad/02_model/{str_variant}/00_preprocessing/01_create_preprocessor/{str_filename}'
    str_local_path = f'./{str_filename}'
    download_from_s3(
        str_local_path=str_local_path, 
        str_bucket_path=str_bucket_path, 
        str_project=str_project,
    )
# load
str_filename = 'cls_model_preprocessing.pkl'
str_local_path = f'./{str_filename}'
cls_model_preprocessing = pickle.load(open(str_local_path, 'rb'))
os.remove(str_local_path)
df_tmp = cls_model_preprocessing.transform(df_tmp)
os.remove('./preprocessing.py')
df_tmp

NaN Replacer: 0.54211 sec.


100%|██████████| 3/3 [00:00<00:00, 135.57it/s]

Set strings: 0.024921 sec.


Boolean Replacer: 0.63181 sec.


100%|██████████| 2478/2478 [00:00<00:00, 2700.14it/s]


...


/home/ec2-user/SageMaker/20231010_gen_xii/11_monitoring/08_logistic_regression/preprocessing.py:442: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['ENG-payment_to_income'] = X['payment__app'] / X['fltgrossmonthly__income_sum']
/home/ec2-user/SageMaker/20231010_gen_xii/11_monitoring/08_logistic_regression/preprocessing.py:447: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['ENG-loan_to_value'] = X['amtfinanced__app'] / X['bookvalue__app']
/home/ec2-user/SageMaker/20231010_gen_xii/11_monitoring/08_logistic_regression/preproces

Replace inf and -inf with NaN: 1.6877 sec.
Imputer: 0.50991 sec.
Map term: 0.013129 sec.
Map PTI: 0.012952 sec.


100%|██████████| 9/9 [00:00<00:00, 1848.89it/s]

Round values: 0.0083242 sec.
Preprocessing Model: 7.7049 sec.


,dtmstampcreation__base,dtmapproved__base,dtmdeclined__base,observationdate__base,analyticsmatchkey__base,decision_dte__base,booked__base,acct_typ_cde__base,open_dte__base,curr_bal_amt__base,...,bitgap__app,dealerstampcreation__app,year,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-payment_to_income,ENG-loan_to_value,ENG-vehicle_age,ENG-dealership_age
99452,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1,2016-02-17 12:47:19.233,2016,1.219360,11,4,0.15,1.000000,-1.0,0.753425
99453,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1,2011-04-12 13:36:22.680,2016,1.219360,11,4,0.09,1.518519,1.0,5.608219
99454,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,2008-01-15 08:53:13.480,2016,1.219360,11,4,0.12,1.409091,4.0,8.849315
99455,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1,2009-06-11 17:08:59.937,2016,1.219360,11,4,0.12,1.281250,3.0,7.443836
99456,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,2012-01-10 16:05:32.420,2016,1.219360,11,4,0.12,1.363636,3.0,4.860274
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124312,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1,2010-07-30 13:50:37.207,2017,1.193925,10,4,0.09,1.016667,-1.0,7.249315
124311,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1,2010-07-30 13:50:37.207,2017,1.193925,10,4,0.09,1.016667,-1.0,7.249315
124313,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1,2013-07-24 10:29:42.193,2017,1.193925,10,4,0.06,1.500000,2.0,4.263014
124314,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,2006-09-09 11:59:48.000,2017,1.193925,10,4,0.15,1.366667,1.0,11.139726


### Generate predictions

In [11]:
# get model
str_filename = 'final_model.pkl'
str_bucket_path = f'02_pricing_pd/02_model/{str_variant}/03_final_model/{str_filename}'
str_local_path = f'./{str_filename}'
download_from_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)
# get inference model
cls_model_inference = pickle.load(open(str_local_path, 'rb'))['model_inference']
# rm
os.remove(str_local_path)
# features
list_cols_model = list(cls_model_inference.feature_names_)
df_tmp['yhat_pricing_pd'] = cls_model_inference.predict_proba(df_tmp[list_cols_model])[:,1]
# sort
df_tmp.sort_values(by=['bigaccountid__app','bitdebtor__app'], ascending=[True,False], inplace=True)
# agg
df_tmp = df_tmp.groupby(by='bigaccountid__app', as_index=False).agg({
    'yhat_pricing_pd': 'mean',
    'target': 'first',
})
# show
df_tmp

/tmp/ipykernel_20087/3039525898.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_tmp['yhat_pricing_pd'] = cls_model_inference.predict_proba(df_tmp[list_cols_model])[:,1]


,bigaccountid__app,yhat_pricing_pd,target
0,2867541.0,0.891058,1
1,2867543.0,0.359111,0
2,2867556.0,0.657558,1
3,2867569.0,0.613629,1
4,2867589.0,0.725824,1
...,...,...,...
19768,3396623.0,0.165674,0
19769,3397156.0,0.067196,0
19770,3397435.0,0.112210,0
19771,3397518.0,0.441041,1


### Join

In [12]:
df = pd.merge(
    left=df,
    right=df_tmp,
    on='bigaccountid__app',
    how='inner',
)
# show
df

,bigaccountid__app,yhat_DQ15_2,DQ15_2,yhat_pricing_pd,target
0,2867541,0.891058,1,0.891058,1
1,2867543,0.359111,0,0.359111,0
2,2867556,0.657558,0,0.657558,1
3,2867569,0.613629,0,0.613629,1
4,2867589,0.725824,0,0.725824,1
...,...,...,...,...,...
19768,3396623,0.165674,0,0.165674,0
19769,3397156,0.067196,0,0.067196,0
19770,3397435,0.112210,0,0.112210,0
19771,3397518,0.441041,0,0.441041,1


### Prepare Data for Logisic Regression models

In [13]:
%%time

# train test split
df_train, df_test = train_test_split(
    df=df, 
    flt_prop_train=0.66,
)
# show
df_train

CPU times: user 14 ms, sys: 506 µs, total: 14.5 ms
Wall time: 13.8 ms


,bigaccountid__app,yhat_DQ15_2,DQ15_2,yhat_pricing_pd,target,row
0,2867541,0.891058,1,0.891058,1,0.000051
1,2867543,0.359111,0,0.359111,0,0.000101
2,2867556,0.657558,0,0.657558,1,0.000152
3,2867569,0.613629,0,0.613629,1,0.000202
4,2867589,0.725824,0,0.725824,1,0.000253
...,...,...,...,...,...,...
13045,3189070,0.362519,0,0.362519,1,0.659789
13046,3189074,0.243164,0,0.243164,1,0.659839
13047,3189078,0.478772,0,0.478772,0,0.659890
13048,3189085,0.497970,0,0.497970,1,0.659940


In [14]:
# show
df_test

,bigaccountid__app,yhat_DQ15_2,DQ15_2,yhat_pricing_pd,target,row
13050,3189100,0.441762,0,0.441762,0,0.660041
13051,3189127,0.330022,0,0.330022,0,0.660092
13052,3189130,0.281791,0,0.281791,1,0.660143
13053,3189173,0.493633,0,0.493633,0,0.660193
13054,3189196,0.345411,0,0.345411,1,0.660244
...,...,...,...,...,...,...
19768,3396623,0.165674,0,0.165674,0,0.999798
19769,3397156,0.067196,0,0.067196,0,0.999848
19770,3397435,0.112210,0,0.112210,0,0.999899
19771,3397518,0.441041,0,0.441041,1,0.999949


In [15]:
# x,y split
list_cols = [
    'yhat_pricing_pd',
    'yhat_DQ15_2',
    'DQ15_2',
]
# train
X_train = df_train[list_cols].copy()
y_train = df_train[str_target]
# test
X_test = df_test[list_cols].copy()
y_test = df_test[str_target]

### Build Logisic Regression models

In [16]:
list_dict_row = []
list_str_trans = [
    'None',
    'Log',
    'Log10',
    'Square',
    'Square_Root',
    'Cube',
    'Cube_Root',
    'Reciprocal',
]
# all combos
for str_trans_a in tqdm(list_str_trans):
    for str_trans_b in list_str_trans:
        # make df copies
        X_train_tmp = X_train.copy()
        X_test_tmp = X_test.copy()

        # get the combination
        dict_combo = {
            'yhat_pricing_pd': str_trans_a,
            'yhat_DQ15_2': str_trans_b,
        }
        #print(dict_combo)

        # apply transformations
        for str_col, str_trans in dict_combo.items():
            if str_trans == 'None':
                X_train_tmp[str_col] = X_train_tmp[str_col]
                X_test_tmp[str_col] = X_test_tmp[str_col]
            elif str_trans == 'Log':
                X_train_tmp[str_col] = np.log(X_train_tmp[str_col])
                X_test_tmp[str_col] = np.log(X_test_tmp[str_col])
            elif str_trans == 'Log10':
                X_train_tmp[str_col] = np.log10(X_train_tmp[str_col])
                X_test_tmp[str_col] = np.log10(X_test_tmp[str_col])
            elif str_trans == 'Square':
                X_train_tmp[str_col] = X_train_tmp[str_col] ** 2
                X_test_tmp[str_col] = X_test_tmp[str_col] ** 2
            elif str_trans == 'Square_Root':
                X_train_tmp[str_col] = np.sqrt(X_train_tmp[str_col])
                X_test_tmp[str_col] = np.sqrt(X_test_tmp[str_col])
            elif str_trans == 'Cube':
                X_train_tmp[str_col] = X_train_tmp[str_col] ** 3
                X_test_tmp[str_col] = X_test_tmp[str_col] ** 3
            elif str_trans == 'Cube_Root':
                X_train_tmp[str_col] = np.cbrt(X_train_tmp[str_col])
                X_test_tmp[str_col] = np.cbrt(X_test_tmp[str_col])
            elif str_trans == 'Reciprocal':
                X_train_tmp[str_col] = 1 / X_train_tmp[str_col]
                X_test_tmp[str_col] = 1 / X_test_tmp[str_col]
            else:
                print('ERROR')

        # cols in model
        list_cols_model = [
            'yhat_pricing_pd',
            'yhat_DQ15_2',
            'DQ15_2',
        ]

        # init model
        cls_model = LogisticRegression()

        # fit model
        try:
            cls_model.fit(
                X_train_tmp[list_cols_model], 
                y_train,
            )
        except ValueError as e:
            print(f'Error: {e}')
            continue

        # predict
        y_hat = cls_model.predict_proba(X_test_tmp[list_cols_model])[:, 1]

        # get score
        flt_score = skm.roc_auc_score(
            y_true=y_test, 
            y_score=y_hat,
        )

        # get intercept and betas
        flt_int = cls_model.intercept_[0]

        # get coefficients
        list_coef = cls_model.coef_.tolist()[0]

        # get features
        list_cols = cls_model.feature_names_in_.tolist()

        # get forula
        str_formula = f'{flt_int:0.3f} + {list_coef[0]:0.3f}*{list_cols[0]} + {list_coef[1]:0.3f}*{list_cols[1]} + {list_coef[2]:0.3f}*{list_cols[2]}'

        # dict_row
        dict_row = {
            'transformation_pricing': str_trans_a,
            'transformation_dq': str_trans_b,
            'score': flt_score,
            'formula': str_formula,
        }
        # append
        list_dict_row.append(dict_row)

# make df
df_results = pd.DataFrame(list_dict_row)
# sort
df_results.sort_values(by='score', ascending=False, inplace=True)
# show
df_results

100%|██████████| 8/8 [00:03<00:00,  2.36it/s]


,transformation_pricing,transformation_dq,score,formula
45,Cube,Cube,0.723463,-1.348 + 2.790*yhat_pricing_pd + 2.790*yhat_DQ...
27,Square,Square,0.723345,-1.648 + 2.309*yhat_pricing_pd + 2.309*yhat_DQ...
29,Square,Cube,0.723333,-1.745 + 6.553*yhat_pricing_pd + -2.639*yhat_D...
43,Cube,Square,0.723333,-1.745 + -2.639*yhat_pricing_pd + 6.553*yhat_D...
0,None,None,0.723010,-2.465 + 2.131*yhat_pricing_pd + 2.131*yhat_DQ...
...,...,...,...,...
15,Log,Reciprocal,0.722303,0.811 + 1.642*yhat_pricing_pd + 0.037*yhat_DQ1...
18,Log10,Log10,0.722291,0.776 + 1.717*yhat_pricing_pd + 1.717*yhat_DQ1...
10,Log,Log10,0.722276,0.781 + 1.259*yhat_pricing_pd + 0.547*yhat_DQ1...
17,Log10,Log,0.722276,0.781 + 0.547*yhat_pricing_pd + 1.259*yhat_DQ1...
